In [3]:
import sliced_mw as smw
import GMM_utils as GMM
from tqdm import tqdm
import pandas as pd
import numpy as np
import time
import torch

np.random.seed(42)
torch.manual_seed(42)

K_ls = [10, 100, 500]
D_ls = [10, 100, 500]
iternum = 10

d = {}
d_std = {}

for K in K_ls:
    d[K] = {}
    d_std[K] = {}
    for D in tqdm(D_ls):
        avg_time = 0.
        time_ls = []
        for iter in range(iternum):
            gmm1 = GMM.RandomGaussianMixtureModel(K, D, device="cpu")
            gmm2 = GMM.RandomGaussianMixtureModel(K, D, device="cpu")
            t0 = time.time()
            _ = smw.calc_SMSW(gmm1, gmm2, pnum=int(1e5), threshold=1e-3)
            time_ls.append((time.time() - t0))
        d[K][D] = str(np.mean(time_ls).round(2)) + "+-" + str(np.std(time_ls).round(2))
        d_std[K][D] = np.std(time_ls)
        
df = pd.DataFrame.from_dict(d)
print(df.to_latex())
display(df)

#df_std = pd.DataFrame.from_dict(d_std).round(5)
#print(df_std.to_latex())
#display(df_std)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [05:01<00:00, 100.45s/it]

\begin{tabular}{llll}
\toprule
 & 10 & 100 & 500 \\
\midrule
10 & 0.15+-0.05 & 0.02+-0.01 & 0.01+-0.01 \\
100 & 0.13+-0.08 & 0.05+-0.03 & 0.08+-0.04 \\
500 & 0.34+-0.24 & 0.89+-0.41 & 1.05+-0.48 \\
\bottomrule
\end{tabular}



,10,100,500
10,0.15+-0.05,0.02+-0.01,0.01+-0.01
100,0.13+-0.08,0.05+-0.03,0.08+-0.04
500,0.34+-0.24,0.89+-0.41,1.05+-0.48


In [2]:
import time
import sliced_mw as smw
import GMM_utils as GMM
from tqdm import tqdm
import pandas as pd

np.random.seed(42)
torch.manual_seed(42)

K_ls = [10, 100, 500]
D_ls = [10, 100, 500]
iternum = 10


d = {}
d_std = {}

for K in K_ls:
    d[K] = {}
    d_std[K] = {}
    for D in tqdm(D_ls):
        avg_time = 0.
        time_ls = []
        for iter in range(iternum):
            gmm1 = GMM.RandomGaussianMixtureModel(K, D, device="cpu")
            gmm2 = GMM.RandomGaussianMixtureModel(K, D, device="cpu")
            t0 = time.time()
            _ = smw.calc_MSW(gmm1, gmm2, pnum=int(1e5), threshold=1e-3)
            time_ls.append((time.time() - t0))
        d[K][D] = str(np.mean(time_ls).round(2)) + "+-" + str(np.std(time_ls).round(2))
        d_std[K][D] = np.std(time_ls)
        
df = pd.DataFrame.from_dict(d)
print(df.to_latex())
display(df)

#df_std = pd.DataFrame.from_dict(d_std).round(5)
#print(df_std.to_latex())
#display(df_std)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [44:01<00:00, 880.40s/it]


\begin{tabular}{llll}
\toprule
 & 10 & 100 & 500 \\
\midrule
10 & 1.2+-0.14 & 0.86+-0.34 & 1.49+-0.35 \\
100 & 1.15+-0.42 & 2.23+-0.55 & 10.95+-0.72 \\
500 & 4.23+-0.37 & 47.76+-2.2 & 215.95+-5.08 \\
\bottomrule
\end{tabular}



,10,100,500
10,1.2+-0.14,0.86+-0.34,1.49+-0.35
100,1.15+-0.42,2.23+-0.55,10.95+-0.72
500,4.23+-0.37,47.76+-2.2,215.95+-5.08


In [17]:
import time
import sliced_mw as smw
import GMM_utils as GMM
from tqdm import tqdm
import pandas as pd
import numpy as np
import signal
import ot
import scipy.stats as sps
import scipy.linalg as spl
from scipy.optimize import linprog


def GaussianW2(m0,m1,Sigma0,Sigma1):
    # Wasserstein between Gaussians
    # source: https://github.com/judelo/gmmot
    Sigma00  = spl.sqrtm(Sigma0)
    Sigma010 = spl.sqrtm(Sigma00@Sigma1@Sigma00)
    d = np.linalg.norm(m0-m1)**2
    d =+np.trace(Sigma0+Sigma1-2*Sigma010)
    return d

def MW2(pi_0,pi_1,mu_0,mu_1,Sigma0_arr,Sigma1_arr):
    # Return the MW dist
    # source: https://github.com/judelo/gmmot
    K0 = mu_0.shape[0]
    K1 = mu_1.shape[0]
    d  = mu_0.shape[1]
    Sigma0_arr = Sigma0_arr.reshape(K0,d,d)
    Sigma1_arr = Sigma1_arr.reshape(K1,d,d)
    M  = np.zeros((K0,K1))
    
    # Pairwise Wasserstein distance matrix between all Gaussians
    for k in range(K0):
        for l in range(K1):
            M[k,l]  = GaussianW2(mu_0[k,:],mu_1[l,:],Sigma0_arr[k,:,:],Sigma1_arr[l,:,:])
    # Compute OT distance
    wstar     = ot.emd(pi_0,pi_1,M)      
    dist   = np.sum(wstar*M)
    return dist

def calc_MW_org(gmm1, gmm2):
    return MW2(gmm1.weights.numpy(), gmm2.weights.numpy(),
                        gmm1.means.numpy(), gmm2.means.numpy(),
                        gmm1.covariances.numpy(), gmm2.covariances.numpy())
np.random.seed(42)
torch.manual_seed(42)

K_ls = [10, 100, 500]
D_ls = [10, 100, 500]
iternum = 10

d = {}

def timeout_handler(signum, frame):
    """Raises a timeout exception."""
    raise TimeoutError

# Set the signal for timeout
signal.signal(signal.SIGALRM, timeout_handler)

for K in K_ls:
    d[K] = {}
    for D in tqdm(D_ls):
        time_ls = []
        try:
            for iter in range(iternum):
                gmm1 = GMM.RandomGaussianMixtureModel(K, D, device="cpu")
                gmm2 = GMM.RandomGaussianMixtureModel(K, D, device="cpu")
                
                signal.alarm(60)  # Set a 30s timeout
                t0 = time.time()
                try:
                    #_ = smw.calc_MW(gmm1, gmm2)
                    _ = calc_MW_delon(gmm1, gmm2)
                    elapsed_time = time.time() - t0
                    time_ls.append(elapsed_time)
                finally:
                    signal.alarm(0)  # Disable the alarm after execution
                
            d[K][D] = str(np.mean(time_ls).round(3)) + "+-" + str(np.std(time_ls).round(3))
        
        except TimeoutError:
            d[K][D] = ">60"

df = pd.DataFrame.from_dict(d).round(5)
print(df.to_latex())
display(df)


  0%|                                                                                                                                                                                                                                                   | 0/3 [00:00<?, ?it/s]/tmp/ipykernel_25988/1778179331.py:34: ComplexWarning: Casting complex values to real discards the imaginary part
  M[k,l]  = GaussianW2(mu0[k,:],mu1[l,:],S0[k,:,:],S1[l,:,:])
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [10:08<00:00, 202.89s/it]

\begin{tabular}{llll}
\toprule
 & 10 & 100 & 500 \\
\midrule
10 & 0.046+-0.075 & 1.854+-0.017 & 46.432+-0.32 \\
100 & 2.587+-0.191 & >60 & >60 \\
500 & >60 & >60 & >60 \\
\bottomrule
\end{tabular}



,10,100,500
10,0.046+-0.075,1.854+-0.017,46.432+-0.32
100,2.587+-0.191,>60,>60
500,>60,>60,>60
